# Lekcija 01 - Uvod u AI agente

Dobrodošli na prvu lekciju tečaja **AI agenti za početnike**!

**AI agent** je program koji koristi veliki jezični model (LLM) kao svoj mehanizam rezoniranja i može poduzimati *akcije* u stvarnom svijetu — pozivati API-je, upitavati baze podataka ili izvršavati kod — kako bi postigao cilj u ime korisnika.

U ovom bilježniku izgradit ćete svog prvog agenta: **Putničkog agenta** koji preporučuje destinacije za odmor. Usput ćete naučiti kako:

1. Povezati se s Azure AI Foundry Agent Service pomoću **Microsoft Agent Framework**.
2. Dati agentu **alat** — običnu Python funkciju koju može nazvati.
3. Pokrenuti agenta i pregledati njegov odgovor.
4. U stvarnom vremenu pratiti odgovor agenta token po token.


## Postavljanje

Prije pokretanja ovog bilježnika, provjerite da imate:

1. **Azure AI Foundry projekt** s implementiranim chat modelom (npr. `gpt-4o-mini`).
2. **Prijavu u Azure CLI** — pokrenite `az login` u vašem terminalu.
3. **Postavljene potrebne varijable okoline:**
   - `AZURE_AI_PROJECT_ENDPOINT` — vaš Azure AI Foundry endpoint projekta.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašeg implementiranog modela.

Ćelija ispod instalira Python pakete koje trebate.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kreiranje Vašeg Prvog Agenta

Agentu su potrebne dvije stvari:

- **Upute** koje mu govore *tko je* i *kako se ponašati* (sistemski prompt).
- **Alati** — Python funkcije označene s `@tool` koje agent može pozvati za dohvat informacija ili izvođenje radnji.

Ispod definiramo jednostavan alat koji vraća popis popularnih destinacija za odmor. Agent će koristiti ovaj alat kada korisnik zatraži preporuke za putovanja.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Za interaktivnije iskustvo možete **streamati** odgovor agenta. Umjesto da čekate kompletan odgovor, agent izdaje dijelove teksta kako se generiraju. Ovo je posebno korisno u sučeljima za chat gdje želite prikazati izlaz u stvarnom vremenu.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Sažetak

U ovoj lekciji naučili ste kako:

- **Stvoriti pružatelja usluge** koji se povezuje na Azure AI Foundry Agent Service putem `AzureAIProjectAgentProvider`.
- **Definirati alat** koristeći dekorator `@tool` kako bi agent mogao pozivati vaše Python funkcije.
- **Pokrenuti agenta** s korisničkom porukom i ispisati njegov odgovor.
- **Prikazivati odgovore u stvarnom vremenu** za izravan izlaz.

U sljedećoj lekciji detaljnije ćemo istražiti agentske okvire i naučiti kako agentima dati moćnije alate i višestepene sposobnosti zaključivanja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje od odgovornosti**:
Ovaj je dokument preveden korištenjem AI usluge za prijevod [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo osigurati točnost, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na njegovom izvornom jeziku treba smatrati autoritativnim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Ne snosimo odgovornost za bilo kakva nerazumijevanja ili pogrešna tumačenja koja proizlaze iz upotrebe ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
